# Master Episode Pipeline
Enter a `NEW_EPISODE_ID` once — everything runs automatically and outputs one organized ZIP.

**Pipeline steps:**
1. FEMA NFIP Claims
2. USGS High-Water Marks (with on-click photos, where USGS provides them)
3. IFC Inundation Maps (KMZ + GeoPackage)
4. Sensor Time Series (IFC + USGS) — with on-click hydrograph charts (rain+stage combo for hydrostations)
5. NOAA MRMS QPE precipitation (watershed-masked raster + map overlay)
6. Interactive Folium Map
7. **Single ZIP export** — one prompt, one file, organized folders


In [ ]:
import ast, base64, gzip, io, os, re, tempfile, threading, time, zipfile
import matplotlib
matplotlib.use('Agg')  # headless rendering -- no display needed to build
                        # the hydrograph PNGs embedded in map popups
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import folium
import geopandas as gpd
import fiona
import numpy as np
import pandas as pd
import requests
import s3fs      # anonymous access to NOAA's public MRMS S3 archive
import xarray as xr  # reads MRMS GRIB2 grids (requires the 'cfgrib' engine,
                      # which itself needs the system eccodes library --
                      # pip install cfgrib xarray s3fs; if eccodes isn't
                      # already on your system, see:
                      # https://github.com/ecmwf/cfgrib#binary-dependencies
import rioxarray  # registers the .rio accessor used to write georeferenced
                  # GeoTIFFs from MRMS grids -- pip install rioxarray
import rasterio   # used directly (not just via rioxarray) to append the
                  # MRMS raster as an extra table inside the same GeoPackage
                  # that already holds the IFC inundation polygons
from rasterio.transform import from_bounds
import branca.colormap as bcm  # legend/colorbar for the MRMS map overlay
from shapely.geometry import shape as shapely_shape, mapping as shapely_mapping
from shapely.ops import unary_union

fiona.drvsupport.supported_drivers['KML']    = 'rw'
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'
print('✓ Imports complete.')

In [ ]:
NOAA_EVENTS_FILE       = 'noaa_21-25_with_huc_08.csv'
IFC_SENSORS_FILE       = 'sensor_csv_huc08/ifis_stream_sensors_huc08.csv'
IFC_HYDROSTATIONS_FILE = 'sensor_csv_huc08/ifis_hydrostations_huc08.csv'
USGS_SENSORS_FILE      = 'sensor_csv_huc08/usgs_stream_sensors_huc08_with_foreign_id1.csv'
DEFAULT_EPISODE_ID     = '191899_0'

MAX_WORKERS         = 6
MAX_RETRIES         = 2
RETRY_DELAY_SECONDS = 4

# ── IFC community lookup ─────────────────────────────────────────────────────
# IFIS internal community IDs are undocumented and not derivable from
# county/city names. Rather than hand-guessing them (a past guess, Cedar
# Falls=503, turned out to point at an empty placeholder record), we
# auto-discover and cache a verified {COMMUNITY NAME: ifis_id} lookup —
# see discover_ifis_communities() / load_ifis_community_lookup() below.
IFIS_COMMUNITY_LOOKUP_FILE = 'ifis_community_ids.csv'

# Some counties contain multiple separately-mapped IFIS communities (e.g.
# Black Hawk County has both Waterloo and Cedar Falls as distinct IFIS
# community records). This widens which community names get checked
# against a county's episode rows. Add to it as you discover more.
COUNTY_TO_CITIES = {
    'BLACK HAWK': ['WATERLOO', 'CEDAR FALLS'],
    'JOHNSON':    ['IOWA CITY'],
    'LINN':       ['CEDAR RAPIDS'],
    'POLK':       ['DES MOINES'],
}

# ── NOAA MRMS QPE (radar-gauge precipitation) ─────────────────────────────
# Public, anonymous S3 archive of NOAA's Multi-Radar Multi-Sensor hourly
# precipitation grids. MultiSensor_QPE_01H_Pass2 is the gauge-corrected
# hourly product (Pass2 = more latency, more gauge correction -- the
# right choice for retrospective evaluation, vs Pass1 which trades
# accuracy for low latency in real-time use). Data available from
# 2020-10-14 onward, so it fully covers a 2021-2025 study period.
MRMS_BUCKET  = 'noaa-mrms-pds'
MRMS_PRODUCT = 'MultiSensor_QPE_01H_Pass2_00.00'

IFC_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap',
}

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────────────────────

def normalize_huc(series):
    return series.fillna('').astype(str).str.replace(r'\.0$','',regex=True).str.strip().str.zfill(8)

def load_noaa_events(csv_path=NOAA_EVENTS_FILE):
    df = pd.read_csv(csv_path)
    df.columns = [c.upper().strip() for c in df.columns]
    df['FIPS_5']    = df['STATE_FIPS'].astype(str).str.zfill(2) + df['CZ_FIPS'].astype(str).str.zfill(3)
    df['HUC8_clean']= normalize_huc(df['HUC8'])
    df['BEGIN_DT']  = pd.to_datetime(df['BEGIN_DATE_TIME'])
    df['END_DT']    = pd.to_datetime(df['END_DATE_TIME'])
    return df

def get_lat_lon(row, lat_candidates, lon_candidates):
    """Robustly extracts lat/lon from a sensor row by trying multiple column names."""
    lat, lon = None, None
    for c in lat_candidates:
        v = row.get(c)
        if v is not None and pd.notna(v):
            try:
                lat = float(v)
                break
            except (ValueError, TypeError):
                pass
    for c in lon_candidates:
        v = row.get(c)
        if v is not None and pd.notna(v):
            try:
                lon = float(v)
                break
            except (ValueError, TypeError):
                pass
    return lat, lon

LAT_COLS = ['lat','latitude','LAT','LATITUDE','y','Y','lat_dd','dec_lat_va']
LON_COLS = ['lon','longitude','LON','LONGITUDE','x','X','lon_dd','lng','dec_long_va']

print('✓ Shared helpers defined.')

✓ Shared helpers defined.


In [ ]:
# ── FEMA helpers ───────────────────────────────────────────────────────────────

def query_fema_claims_multi_fips(fips_list, start_date, end_date, date_buffer_days=14):
    base_url     = 'https://www.fema.gov/api/open/v2/FimaNfipClaims'
    search_start = (start_date - timedelta(days=2)).strftime('%Y-%m-%d')
    search_end   = (end_date   + timedelta(days=date_buffer_days)).strftime('%Y-%m-%d')
    fips_cond    = ' or '.join([f"countyCode eq '{f}'" for f in fips_list])
    fema_filter  = (f'({fips_cond}) and '
                    f'dateOfLoss ge {search_start}T00:00:00.000Z and '
                    f'dateOfLoss le {search_end}T23:59:59.000Z')
    try:
        r = requests.get(base_url, params={'$filter': fema_filter, '$top': 10000})
        r.raise_for_status()
        return pd.DataFrame(r.json().get('FimaNfipClaims', []))
    except Exception as e:
        print(f'Error querying OpenFEMA API: {e}')
        return pd.DataFrame()

print('✓ FEMA helpers defined.')

✓ FEMA helpers defined.


In [ ]:
# ── USGS HWM helpers ───────────────────────────────────────────────────────────

def query_usgs_hwms(min_lat, max_lat, min_lon, max_lon, start_date, end_date, date_buffer_days=30):
    try:
        r = requests.get('https://stn.wim.usgs.gov/STNServices/HWMs.json', timeout=15)
        r.raise_for_status()
        hwms = r.json()
        if not hwms:
            return pd.DataFrame()
        df_hwm  = pd.DataFrame(hwms)
        lat_col = 'latitude_dd'  if 'latitude_dd'  in df_hwm.columns else ('latitude'  if 'latitude'  in df_hwm.columns else None)
        lon_col = 'longitude_dd' if 'longitude_dd' in df_hwm.columns else ('longitude' if 'longitude' in df_hwm.columns else None)
        if not lat_col or not lon_col:
            print('Warning: coordinate columns not found in USGS response.')
            return pd.DataFrame()
        for col in ['flagDate','surveyDate','approvalDate']:
            if col in df_hwm.columns:
                df_hwm['date_parsed'] = pd.to_datetime(df_hwm[col], errors='coerce')
                break
        pad = 0.1
        mask_sp = ((df_hwm[lat_col] >= min_lat-pad) & (df_hwm[lat_col] <= max_lat+pad) &
                   (df_hwm[lon_col] >= min_lon-pad) & (df_hwm[lon_col] <= max_lon+pad))
        if 'date_parsed' in df_hwm.columns:
            s = start_date - timedelta(days=2)
            e = end_date   + timedelta(days=date_buffer_days)
            mask_t = (df_hwm['date_parsed'] >= s) & (df_hwm['date_parsed'] <= e)
            return df_hwm[mask_sp & mask_t]
        return df_hwm[mask_sp]
    except Exception as e:
        print(f'Error querying USGS STN API: {e}')
        return pd.DataFrame()

def get_hwm_photo_urls(row, max_photos=3):
    """
    Extracts photo URLs from a HWM record's nested 'files' field, if
    present. This field's exact structure hasn't been directly verified
    against a live STNServices/HWMs.json response (unlike most other
    fields in this pipeline) -- it's built from USGS's own 'stormevents'
    Python library docs and the STN CSV field list, both of which show a
    'files' array per HWM with file_id + filename entries, plus the
    confirmed-working file URL pattern
    https://stn.wim.usgs.gov/STNServices/Files/{file_id}/Item.
    Only entries with an image-like extension are returned -- PDFs and
    other file types (field notes, recovery forms) are skipped since
    they can't render as an <img> tag. Returns [] if no 'files' field is
    found at all; that's a real possibility given the unverified schema,
    not necessarily a sign anything is broken -- also many HWMs simply
    have no photos.
    """
    files = row.get('files')
    if files is None or (isinstance(files, float) and pd.isna(files)):
        return []
    if not isinstance(files, list):
        return []
    image_exts = ('.jpg', '.jpeg', '.png', '.gif')
    urls = []
    for f in files:
        if not isinstance(f, dict):
            continue
        name = str(f.get('name', '')).lower()
        file_id = f.get('file_id') or f.get('fileId') or f.get('id')
        if file_id and name.endswith(image_exts):
            urls.append(f'https://stn.wim.usgs.gov/STNServices/Files/{file_id}/Item')
        if len(urls) >= max_photos:
            break
    return urls

print('✓ USGS HWM helpers defined.')

In [ ]:
# ── NOAA MRMS QPE helpers ─────────────────────────────────────────────────────

_mrms_fs = None
def _get_mrms_fs():
    """Lazily creates a single shared anonymous S3 filesystem handle."""
    global _mrms_fs
    if _mrms_fs is None:
        _mrms_fs = s3fs.S3FileSystem(anon=True)
    return _mrms_fs

# eccodes (the C library cfgrib wraps) is not guaranteed thread-safe.
# Running the S3 download/decompress in parallel is fine -- that's pure
# Python/network work -- but calling xr.load_dataset(engine='cfgrib')
# concurrently from multiple threads can corrupt the GRIB decode itself
# (seen as 'Decoding invalid' / 'No valid message found' errors, on
# different files, happening together). This lock forces the actual
# decode step to run one-at-a-time while still letting downloads overlap.
_cfgrib_lock = threading.Lock()

def fetch_mrms_hour(dt_utc, verbose=True):
    """
    Fetches one hourly MRMS MultiSensor_QPE_01H_Pass2 GRIB2 grid for the
    UTC hour containing dt_utc, from NOAA's public S3 archive
    (s3://noaa-mrms-pds). Returns an xarray.DataArray of 1-hour accumulated
    precipitation in mm on the native ~1km CONUS grid (longitude stored as
    0-360, not -180/180), or None if no file is found for that hour.

    IMPORTANT: dt_utc must already be in UTC. MRMS filenames are UTC-based;
    if your source date/time is local (e.g. NOAA Storm Events
    BEGIN_DATE_TIME, which is local standard time, not UTC), convert to
    UTC before calling this -- see the timezone-conversion block in the
    main pipeline step for how that's handled here.
    """
    fs = _get_mrms_fs()
    date_str = dt_utc.strftime('%Y%m%d')
    hour_str = dt_utc.strftime('%H')
    prefix  = f'{MRMS_BUCKET}/CONUS/{MRMS_PRODUCT}/{date_str}/'
    pattern = f'{prefix}MRMS_{MRMS_PRODUCT}_{date_str}-{hour_str}*.grib2.gz'

    tmp_path = None
    try:
        matches = sorted(fs.glob(pattern))
        if not matches:
            if verbose:
                print(f'  [-] No MRMS file found for {dt_utc.strftime("%Y-%m-%d %H:00")} UTC')
            return None
        key = matches[0]
        with fs.open(key, 'rb') as gz_file:
            raw = gzip.decompress(gz_file.read())

        # Write to a temp file and fully close it before letting cfgrib/
        # eccodes (a C library) open the same path. Using
        # NamedTemporaryFile's own 'with' block here would leave the file
        # locked on Windows, which blocks that second open and causes a
        # PermissionError -- this doesn't happen on Linux/Mac, which allow
        # concurrent opens of the same file.
        fd, tmp_path = tempfile.mkstemp(suffix='.grib2')
        with os.fdopen(fd, 'wb') as tmp_file:
            tmp_file.write(raw)

        # load_dataset (not open_dataset) reads eagerly into memory, so
        # it's safe to delete the temp file right after this call returns.
        # indexpath='' skips cfgrib's own .idx sidecar cache file -- not
        # useful here since every temp file is unique and deleted right
        # after, and it's one more thing that can fail on Windows.
        with _cfgrib_lock:
            ds = xr.load_dataset(
                tmp_path, engine='cfgrib', decode_timedelta=False,
                backend_kwargs={'indexpath': ''}
            )
        var_name = list(ds.data_vars)[0]
        da = ds[var_name]
        # MRMS uses large negative sentinel values to mark missing/no-coverage
        # cells rather than NaN -- mask those out before any aggregation.
        da = da.where(da >= 0)
        return da
    except Exception as e:
        print(f'  [-] Error fetching MRMS for {dt_utc}: {e}')
        return None
    finally:
        # Clean up the temp .grib2 file AND the .idx sidecar file cfgrib
        # creates next to it during reading, so temp dir doesn't accumulate
        # junk across a long per-hour fetch loop.
        if tmp_path and os.path.exists(tmp_path):
            tmp_dir = os.path.dirname(tmp_path)
            tmp_name = os.path.basename(tmp_path)
            for f in os.listdir(tmp_dir):
                if f.startswith(tmp_name):
                    try:
                        os.remove(os.path.join(tmp_dir, f))
                    except OSError:
                        pass

def extract_mrms_bbox_stats(da, min_lat, max_lat, min_lon, max_lon):
    """
    Subsets an MRMS DataArray to a lat/lon bounding box and returns
    (mean_mm, max_mm, n_cells) for that hour. MRMS stores longitude as
    0-360 (not -180 to 180), so bbox longitudes are converted before
    slicing.
    """
    lon_min_360 = min_lon % 360
    lon_max_360 = max_lon % 360
    sub = da.sel(latitude=slice(max_lat, min_lat), longitude=slice(lon_min_360, lon_max_360))
    values = sub.values
    valid = values[~pd.isna(values)]
    if valid.size == 0:
        return None, None, 0
    return float(valid.mean()), float(valid.max()), int(valid.size)

def fetch_mrms_grid_subset(dt_utc, min_lat, max_lat, min_lon, max_lon, verbose=True):
    """
    Like fetch_mrms_hour, but returns the bbox-subsetted grid itself (an
    xarray.DataArray with lat/lon coords, still 0-360 longitude) rather
    than a scalar stat -- used for building the spatial accumulation
    raster, as opposed to the hourly mean/max time series.
    """
    da = fetch_mrms_hour(dt_utc, verbose=verbose)
    if da is None:
        return None
    lon_min_360 = min_lon % 360
    lon_max_360 = max_lon % 360
    sub = da.sel(latitude=slice(max_lat, min_lat), longitude=slice(lon_min_360, lon_max_360))
    if sub.size == 0:
        return None
    return sub

def query_mrms_precipitation(start_date_utc, end_date_utc, min_lat, max_lat, min_lon, max_lon,
                              max_workers=MAX_WORKERS, verbose=True):
    """
    Builds an hourly precipitation time series (mean/max mm per hour over
    the given bbox) for every UTC hour between start_date_utc and
    end_date_utc inclusive. Fetches hours in parallel (each hour pulls a
    full CONUS-wide GRIB2 file, so this is genuinely slow done serially --
    reuses the same ThreadPoolExecutor pattern as the sensor download step).
    Returns a DataFrame with columns: datetime_utc, mean_mm, max_mm, n_cells.
    """
    hours = pd.date_range(
        start_date_utc.replace(minute=0, second=0, microsecond=0),
        end_date_utc.replace(minute=0, second=0, microsecond=0),
        freq='h'
    )

    def _process_hour(hr):
        da = fetch_mrms_hour(hr.to_pydatetime(), verbose=False)
        if da is None:
            return {'datetime_utc': hr, 'mean_mm': None, 'max_mm': None, 'n_cells': 0}
        mean_mm, max_mm, n_cells = extract_mrms_bbox_stats(da, min_lat, max_lat, min_lon, max_lon)
        return {'datetime_utc': hr, 'mean_mm': mean_mm, 'max_mm': max_mm, 'n_cells': n_cells}

    rows = []
    completed = 0
    if verbose:
        print(f'  Fetching {len(hours)} hourly MRMS grids ({max_workers} threads)...')
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_hr = {executor.submit(_process_hour, hr): hr for hr in hours}
        for future in as_completed(future_to_hr):
            rows.append(future.result())
            completed += 1
            if verbose and (completed % 10 == 0 or completed == len(hours)):
                print(f'     ...{completed}/{len(hours)} hours processed')

    df = pd.DataFrame(rows).sort_values('datetime_utc').reset_index(drop=True)
    if verbose:
        found = df['mean_mm'].notna().sum()
        print(f'  [✓] {found}/{len(df)} MRMS hourly grids retrieved for bbox')
    return df

def compute_episode_precip_grid(start_date_utc, end_date_utc, min_lat, max_lat, min_lon, max_lon,
                                 max_workers=MAX_WORKERS, verbose=True):
    """
    Fetches every hourly MRMS grid across the episode window, subsets each
    to the bbox, and sums them into one 'total storm rainfall' grid (mm)
    covering the full episode duration -- for mapping/raster export, as
    opposed to query_mrms_precipitation()'s per-hour scalar time series.

    Returns (total_da, count_da):
      total_da  -- accumulated mm per grid cell across the episode
      count_da  -- number of hours that actually had data per cell (QC --
                   cells with a low count relative to the total hour count
                   had missing hourly grids, so their total is an
                   undercount, not necessarily a dry cell)
    Returns (None, None) if no hours were retrieved at all.
    """
    hours = pd.date_range(
        start_date_utc.replace(minute=0, second=0, microsecond=0),
        end_date_utc.replace(minute=0, second=0, microsecond=0),
        freq='h'
    )
    if verbose:
        print(f'  Fetching {len(hours)} hourly grids for spatial accumulation ({max_workers} threads)...')

    grids = []
    completed = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_hr = {
            executor.submit(fetch_mrms_grid_subset, hr.to_pydatetime(), min_lat, max_lat, min_lon, max_lon, False): hr
            for hr in hours
        }
        for future in as_completed(future_to_hr):
            sub = future.result()
            completed += 1
            if sub is not None:
                grids.append(sub)
            if verbose and (completed % 10 == 0 or completed == len(hours)):
                print(f'     ...{completed}/{len(hours)} hours processed ({len(grids)} with data)')

    if not grids:
        if verbose:
            print('  [-] No MRMS grids retrieved -- cannot build accumulation raster')
        return None, None

    # Align on the first grid's coordinates in case of tiny floating-point
    # coordinate drift between files, then sum across the hour dimension.
    ref = grids[0]
    aligned = [g.reindex_like(ref, method='nearest', tolerance=0.02) for g in grids]
    stacked = xr.concat(aligned, dim='hour')
    total_da = stacked.sum(dim='hour', skipna=True)
    count_da = stacked.count(dim='hour')

    if verbose:
        print(f'  [✓] Accumulated {len(grids)}/{len(hours)} hourly grids into total storm rainfall raster')
    return total_da, count_da

def _grid_to_180_lon(da):
    """Converts an MRMS DataArray's longitude coordinate from 0-360 to
    -180/180 and sorts ascending, for compatibility with standard GIS
    tools (GeoTIFF, GeoPackage, Folium) which all expect -180/180."""
    da = da.copy()
    lon_180 = ((da['longitude'].values + 180) % 360) - 180
    return da.assign_coords(longitude=lon_180).sortby('longitude')

def save_precip_grid_geotiff(total_da, out_path):
    """
    Saves an accumulated MRMS precipitation grid (mm) to a standalone
    GeoTIFF in EPSG:4326. This is the primary, reliable raster output --
    opens in any GIS tool with zero special handling.
    """
    da = _grid_to_180_lon(total_da)
    da = da.rio.write_crs('EPSG:4326')
    da.rio.to_raster(out_path)

def append_precip_grid_to_gpkg(total_da, gpkg_path, table_name='mrms_precip_total_mm', verbose=True):
    """
    Appends the accumulated MRMS precipitation grid as a RASTER table
    inside the same GeoPackage file that holds the IFC inundation
    polygons (GeoPackage supports both vector and raster tables in one
    file). This is a secondary, more fragile path than
    save_precip_grid_geotiff() -- it depends on GDAL's GPKG raster driver
    behaving consistently across platforms/versions, which hasn't been
    tested outside this notebook's own environment. Wrapped so a failure
    here doesn't take down the rest of the export; the standalone GeoTIFF
    is always the safe fallback.
    """
    try:
        da = _grid_to_180_lon(total_da)
        # GeoPackage's raster driver only supports Byte/Int16/UInt16/Float32
        # -- xarray's .sum() upcasts to float64 by default, which GDAL's
        # GPKG driver silently rejects, so this must be cast down before
        # writing (GeoTIFF has no such restriction, which is why
        # save_precip_grid_geotiff doesn't need this cast).
        values = da.values.astype('float32')
        lats = da['latitude'].values
        lons = da['longitude'].values
        transform = from_bounds(lons.min(), lats.min(), lons.max(), lats.max(),
                                 values.shape[1], values.shape[0])
        with rasterio.open(
            gpkg_path, 'w',
            driver='GPKG',
            height=values.shape[0], width=values.shape[1], count=1,
            dtype=values.dtype, crs='EPSG:4326', transform=transform,
            RASTER_TABLE=table_name,
            APPEND_SUBDATASET='YES',
        ) as dst:
            dst.write(values, 1)
        if verbose:
            print(f'  [✓] Raster table \'{table_name}\' appended to {gpkg_path}')
        return True
    except Exception as e:
        print(f'  [-] Could not append raster to GeoPackage (GeoTIFF export is unaffected): {e}')
        return False

def precip_grid_to_overlay(total_da, opacity=180, colors=None):
    """
    Converts an accumulated MRMS precipitation grid into the pieces
    needed for a Folium ImageOverlay: an (H, W, 4) uint8 RGBA array,
    [[south, west], [north, east]] bounds, and a matching branca
    LinearColormap for the map legend. NaN cells are rendered fully
    transparent.
    """
    da = _grid_to_180_lon(total_da)
    values = da.values
    lats = da['latitude'].values
    lons = da['longitude'].values

    # Standard green -> yellow -> orange -> red -> dark-red radar/forecast
    # ramp (light color = little rain, red/dark-red = heaviest), the same
    # family used by NWS precipitation maps -- swap this list for a
    # different palette if you'd rather.
    colors = colors or ['#ffffff', '#a1d99b', '#31a354', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']
    finite_vals = values[~pd.isna(values)]
    vmin = float(finite_vals.min()) if finite_vals.size else 0.0
    vmax = float(finite_vals.max()) if finite_vals.size else 1.0
    if vmax <= vmin:
        vmax = vmin + 1.0
    colormap = bcm.LinearColormap(colors=colors, vmin=vmin, vmax=vmax)

    rgba = np.zeros((*values.shape, 4), dtype=np.uint8)
    finite_mask = ~pd.isna(values)
    for i, j in zip(*np.where(finite_mask)):
        hexcolor = colormap(values[i, j])
        r, g, b = int(hexcolor[1:3], 16), int(hexcolor[3:5], 16), int(hexcolor[5:7], 16)
        rgba[i, j] = [r, g, b, opacity]

    # latitude coord is descending (north -> south), which matches the
    # top-to-bottom row order ImageOverlay expects for a standard image.
    bounds = [[float(lats.min()), float(lons.min())], [float(lats.max()), float(lons.max())]]
    return rgba, bounds, colormap

print('✓ NOAA MRMS QPE helpers defined.')

In [ ]:
# ── IFC Inundation helpers ─────────────────────────────────────────────────────

def get_ifis_kmz_items(ifis_id):
    url = f'https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={ifis_id}'
    try:
        res  = requests.get(url, headers=IFC_HEADERS, timeout=10)
        res.raise_for_status()
        data = ast.literal_eval(res.text)
        items = []
        if len(data) > 2 and data[2] and data[2][0]:
            for mi in data[2][0][0]:
                items.append((mi[0], f'stage_{mi[1]}ft'))
        if len(data) > 2 and len(data[2]) > 1 and data[2][1]:
            for mi in data[2][1][0]:
                items.append((mi[0], f'flood_{mi[3]}yr'))
        return items
    except Exception as e:
        print(f'  [-] IFIS ID {ifis_id}: {e}')
        return []

def convert_kmz_bytes_to_geodataframe(kmz_bytes, extent_name):
    try:
        with zipfile.ZipFile(io.BytesIO(kmz_bytes)) as z:
            kml_files = [f for f in z.namelist() if f.endswith('.kml')]
            if not kml_files:
                return None
            kml_data = z.read(kml_files[0])
        gdf = gpd.read_file(io.BytesIO(kml_data), driver='KML')
        gdf['extent_name'] = extent_name
        if gdf.crs is None or gdf.crs.to_epsg() != 3418:
            gdf = gdf.to_crs(epsg=3418)
        gdf['area_acres'] = gdf.geometry.area / 4046.8564224
        return gdf
    except Exception as e:
        print(f'  [-] KMZ parse failed for {extent_name}: {e}')
        return None

def discover_ifis_communities(id_range=range(1, 700), delay=0.05, verbose=True):
    """Scans IFIS numeric community IDs and builds a verified {name: id}
    lookup by reading back the community name IFIS echoes for each ID.
    Only keeps IDs that both have a name AND have at least one real
    stage/frequency map (empty placeholder records, like the old hardcoded
    503 for 'Cedar Falls', are skipped)."""
    found = {}
    for i in id_range:
        url = f'https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={i}'
        try:
            res = requests.get(url, headers=IFC_HEADERS, timeout=10)
            res.raise_for_status()
            data = ast.literal_eval(res.text)
            name = data[0][0].strip().upper() if data and data[0] and data[0][0] else ''
            has_maps = len(data) > 2 and data[2] and data[2][0] and data[2][0][0]
            if name and has_maps:
                found[name] = i
                if verbose:
                    print(f'  [{i}] {name}')
        except Exception:
            pass
        time.sleep(delay)
    return found

def load_ifis_community_lookup(cache_file=IFIS_COMMUNITY_LOOKUP_FILE, id_range=range(1, 700)):
    """Loads the cached {name: ifis_id} lookup if present, otherwise runs
    discover_ifis_communities() once (this takes a few minutes — it's a
    one-time scan) and caches the result to cache_file for future runs."""
    if os.path.exists(cache_file):
        s = pd.read_csv(cache_file, index_col=0).iloc[:, 0]
        return s.to_dict()
    print(f'  [*] No cached lookup at {cache_file} — discovering IFIS community IDs '
          f'(one-time, ~{len(id_range)} requests, a few minutes)...')
    lookup = discover_ifis_communities(id_range=id_range)
    pd.Series(lookup, name='ifis_id').to_csv(cache_file)
    print(f'  [\u2713] Cached {len(lookup)} communities to {cache_file}')
    return lookup

def match_communities_for_episode(episode_rows, community_lookup, county_to_cities=None):
    """Returns (matched_ids, matched_names) relevant to an episode by
    matching known community names (word-boundary, not naive substring)
    against each row's county name (CZ_NAME) and event narrative, then
    widening county matches to any known cities within that county via
    county_to_cities."""
    county_to_cities = county_to_cities or {}
    candidate_names  = set(community_lookup.keys())
    matched_ids      = set()
    matched_names    = set()

    for _, row in episode_rows.iterrows():
        cz   = str(row.get('CZ_NAME', '')).upper().strip()
        narr = str(row.get('EVENT_NARRATIVE', '')).upper()
        text = f'{cz} {narr}'

        # Direct community-name matches, word-boundary (avoids e.g. 'ADEL'
        # falsely matching inside an unrelated word in the narrative).
        for name in candidate_names:
            if re.search(rf'\b{re.escape(name)}\b', text):
                matched_ids.add(community_lookup[name])
                matched_names.add(name)

        # County -> known-cities widening (e.g. Black Hawk -> Waterloo + Cedar Falls).
        county_key = cz.replace(' CO.', '').replace(' COUNTY', '').strip()
        for cities in (county_to_cities.get(county_key, []), county_to_cities.get(cz, [])):
            for city in cities:
                if city in community_lookup:
                    matched_ids.add(community_lookup[city])
                    matched_names.add(city)

    return matched_ids, matched_names

print('\u2713 IFC inundation helpers defined.')

In [ ]:
# ── Sensor pipeline helpers ────────────────────────────────────────────────────

def resolve_hydrostation_id(row):
    for col in ['foreign_id1','ifc_id','station_id','hydro_id','ifis_id','ifc_id.1','Sensor_ID']:
        val = row.get(col)
        if pd.notna(val):
            try:
                n = int(float(val))
                if n > 4000: return str(n)
            except ValueError: pass
    raw = row.get('id')
    if pd.notna(raw):
        try: return str(int(float(raw)))
        except ValueError: pass
    return None

def fetch_nws_lid_data(nws_id, start_dt, end_dt):
    nws_id = str(nws_id).strip().upper()
    try:
        res = requests.get(f'https://api.water.noaa.gov/v1/gauges/{nws_id}/stageflow/observed',
                           headers={'User-Agent':'Mozilla/5.0','Accept':'application/json'}, timeout=12)
        if res.status_code == 200:
            records = []
            for obs in res.json().get('data',[]):
                t = pd.to_datetime(obs.get('validTime'))
                if t.tzinfo: t = t.tz_localize(None)
                if start_dt <= t <= end_dt:
                    records.append({'datetime':t,'parameter':'Stage / Observed','value':obs.get('primary'),'unit':obs.get('primaryUnit','ft')})
            if records:
                df = pd.DataFrame(records).sort_values('datetime')
                print(f'      ✅ {len(df)} records for NWS LID {nws_id}')
                return df
    except Exception: pass
    return pd.DataFrame()

def fetch_ifc_sensor_data(sensor_row, start_dt, end_dt):
    start_str = start_dt.strftime('%Y%m%d'); end_str = end_dt.strftime('%Y%m%d')
    candidates = [c for c in [sensor_row.get('id'), sensor_row.get('foreign_id')] if c is not None]
    for c in candidates:
        try:
            res = requests.get(f'https://hydroiowa.org/api/riversensor/{c}/data/{start_str}/{end_str}', timeout=15)
            if res.status_code == 200:
                observed = res.json().get('observed',[])
                if observed:
                    df = pd.DataFrame(observed)
                    if 'validTime' in df.columns:
                        df['validTime'] = pd.to_datetime(df['validTime'],format='mixed',errors='coerce')
                        df = df.dropna(subset=['validTime'])
                        if df['validTime'].dt.tz is not None: df['validTime'] = df['validTime'].dt.tz_localize(None)
                        df = df[(df['validTime']>=start_dt)&(df['validTime']<=end_dt)].copy()
                    if not df.empty:
                        print(f'      ✅ {len(df)} stage records for IFC sensor {c}')
                        return df
        except Exception: continue
    print(f"      ℹ️ No data for IFC sensor {sensor_row.get('foreign_id')}")
    return pd.DataFrame()

def fetch_ifc_hydrostation_data(station_row, start_dt, end_dt):
    start_str = start_dt.strftime('%Y%m%d'); end_str = end_dt.strftime('%Y%m%d')
    nid = resolve_hydrostation_id(station_row)
    name = station_row.get('foreign_id1')
    if not nid:
        print(f'      ❌ Skipping {name}: no numeric ID')
        return pd.DataFrame()
    hdrs = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    params = ['rain','wind','soil','well','groundwell','stage']
    all_frames = []
    for url in [f"https://hydroiowa.org/api/hydrostation/{nid}/data/{start_str}/{end_str}/?{'&'.join(params)}",
                 f'https://hydroiowa.org/api/hydrostation/{nid}/data/{start_str}/{end_str}']:
        if all_frames: break
        try:
            res = requests.get(url, headers=hdrs, timeout=15)
            if res.status_code == 200:
                obs = res.json().get('observed',{})
                if isinstance(obs, dict):
                    for k,v in obs.items():
                        if isinstance(v,list) and v:
                            p = pd.DataFrame(v); p['parameter_type']=k; all_frames.append(p)
                elif isinstance(obs,list) and obs:
                    all_frames.append(pd.DataFrame(obs))
        except Exception: pass
    if not all_frames:
        print(f'      ℹ️ No data for hydrostation {nid} ({name})')
        return pd.DataFrame()
    df = pd.concat(all_frames, ignore_index=True)
    if 'validTime' in df.columns:
        df['validTime'] = pd.to_datetime(df['validTime'],format='mixed',errors='coerce')
        df = df.dropna(subset=['validTime'])
        if df['validTime'].dt.tz is not None: df['validTime'] = df['validTime'].dt.tz_localize(None)
        df = df[(df['validTime']>=start_dt)&(df['validTime']<=end_dt)]
    if not df.empty: print(f'      ✅ {len(df)} records for hydrostation {nid} ({name})')
    else: print(f'      ℹ️ No records in range for hydrostation {nid} ({name})')
    return df

def fetch_usgs_sensor_data(usgs_row, start_dt, end_dt):
    candidates, nws_lids = [], []
    for key in ['foreign_id1','foreign_id','usgs_site_no_revised','usgs_site_no','id']:
        val = usgs_row.get(key)
        if pd.notna(val):
            cv = str(val).split('.')[0].strip().upper()
            if cv and cv != 'NAN' and cv not in candidates:
                candidates.append(cv)
                if not cv.isdigit(): nws_lids.append(cv)
    if not candidates:
        print('      ❌ Skipping USGS row: no identifier')
        return pd.DataFrame()
    strategies = []
    for sid in candidates:
        if sid.isdigit(): strategies += [('sites',sid.zfill(8)),('nwsLids',sid)]
        else:             strategies += [('nwsLids',sid),('sites',sid)]
    url = 'https://waterservices.usgs.gov/nwis/iv/'
    hdrs = {'User-Agent':'Mozilla/5.0 HydrologicalPipeline/2.0'}
    for param_type, qval in strategies:
        for pcd in ['00065','00060',None]:
            params = {'format':'json', param_type:qval,
                      'startDT':start_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
                      'endDT':  end_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')}
            if pcd: params['parameterCd'] = pcd
            try:
                res = requests.get(url, params=params, headers=hdrs, timeout=15)
                if res.status_code != 200: continue
                records = []
                for ts in res.json().get('value',{}).get('timeSeries',[]):
                    pname = ts.get('variable',{}).get('variableName','Unknown')
                    unit  = ts.get('variable',{}).get('unit',{}).get('unitCode','N/A')
                    for v in ts.get('values',[{}])[0].get('value',[]):
                        records.append({'datetime':v.get('dateTime'),'parameter':pname,'value':v.get('value'),'unit':unit})
                if records:
                    df = pd.DataFrame(records)
                    print(f'      ✅ {len(df)} records for USGS {qval}')
                    # Best-effort: if the primary hit was discharge (00060)
                    # or stage (00065), also try fetching the OTHER of that
                    # pair for the same site, so a dashed stage-height line
                    # can be added alongside flow on the map popup chart.
                    # This is purely additive and wrapped so any failure
                    # here (including USGS not having that parameter at
                    # this site) silently falls through to returning the
                    # already-successful primary df unchanged -- it can
                    # never turn a working fetch into a failed one.
                    other_pcd = {'00060': '00065', '00065': '00060'}.get(pcd)
                    if other_pcd:
                        try:
                            other_params = dict(params, parameterCd=other_pcd)
                            other_res = requests.get(url, params=other_params, headers=hdrs, timeout=15)
                            if other_res.status_code == 200:
                                other_records = []
                                for ts in other_res.json().get('value', {}).get('timeSeries', []):
                                    pname2 = ts.get('variable', {}).get('variableName', 'Unknown')
                                    unit2  = ts.get('variable', {}).get('unit', {}).get('unitCode', 'N/A')
                                    for v in ts.get('values', [{}])[0].get('value', []):
                                        other_records.append({'datetime': v.get('dateTime'), 'parameter': pname2,
                                                               'value': v.get('value'), 'unit': unit2})
                                if other_records:
                                    df = pd.concat([df, pd.DataFrame(other_records)], ignore_index=True)
                        except Exception:
                            pass
                    return df
            except Exception: continue
    for lid in nws_lids:
        df_n = fetch_nws_lid_data(lid, start_dt, end_dt)
        if df_n is not None and not df_n.empty: return df_n
    print(f"      ℹ️ No data for site {candidates[0]}")
    return pd.DataFrame()

def fetch_with_retry(fetch_fn, fetch_args):
    last_exc = None
    for attempt in range(1, MAX_RETRIES+1):
        try:
            df = fetch_fn(*fetch_args)
            if df is not None and not df.empty: return df
        except Exception as exc: last_exc = exc
        if attempt < MAX_RETRIES: time.sleep(RETRY_DELAY_SECONDS)
    if last_exc: raise last_exc
    return pd.DataFrame()

def get_sensor_code(row, sensor_type):
    """
    Canonical sensor identifier, used consistently for (a) keying the
    downloaded time-series dict, (b) naming CSVs in the export ZIP, and
    (c) looking up the right hydrograph for a map popup. Previously the
    map popups and the ZIP-export step computed slightly different ID
    fallback chains for the same sensor (e.g. IFC river sensor popups
    used 'foreign_id' while the export step used 'foreign_id1') -- this
    unifies both to the export step's original logic, since that's what
    actually determines the CSV filenames you already have.
    """
    if sensor_type == 'ifc_river':
        return str(row.get('foreign_id1', row.get('id', 'sensor')))
    elif sensor_type == 'ifc_hydrostation':
        return str(row.get('foreign_id1', row.get('id', 'station'))).replace(' ', '_')
    elif sensor_type == 'usgs':
        return str(row.get('foreign_id1', row.get('usgs_site_no_revised',
               row.get('foreign_id', row.get('id', 'usgs'))))).split('.')[0].strip().upper()
    return str(row.get('id', 'unknown'))

# USGS and NWS-LID fallback data always have known columns ('datetime',
# 'value'), built explicitly in fetch_usgs_sensor_data/fetch_nws_lid_data
# above. IFC river sensor and hydrostation data pass through whatever raw
# JSON keys hydroiowa.org returns beyond 'validTime', which hasn't been
# directly verified against a live response -- VALUE_COL_CANDIDATES tries
# the likely names first, then falls back to guessing the first numeric
# column and printing a warning, so a wrong guess is visible rather than
# silently wrong.
TIME_COL_CANDIDATES  = ['validTime', 'datetime']
# 'accum' confirmed live for IFC hydrostation 'rain' records (accumulated
# precipitation per interval). The rest are unverified guesses for other
# sensor/parameter types -- still checked first since a confirmed/plausible
# name beats a blind heuristic, but treat any hit outside 'accum'/'value'/
# 'datetime'-paired-USGS data as unverified until seen in real output.
VALUE_COL_CANDIDATES = ['value', 'accum', 'stage', 'stage_ft', 'waterlevel', 'water_level', 'reading', 'obs_value']

def _detect_time_value_cols(df, verbose=True):
    """
    Finds the time and value columns in a downloaded sensor DataFrame.
    A candidate column only counts if it actually HAS data in this
    particular subset, not merely exists as a column name -- IFC
    hydrostation data concatenates every parameter type (rain, wind,
    soil, well, stage) into one wide DataFrame, so a 'wind' subset still
    carries rain-only columns like 'accum' as a column, just entirely
    NaN for wind rows. Without this check, a confirmed-real name for one
    parameter type (e.g. 'accum' for rain) would wrongly get picked for
    an unrelated parameter type just because the column exists at all.
    Falls back to the least-NaN numeric column when no named candidate
    has real data, rather than blindly the first numeric column.
    """
    time_col = next((c for c in TIME_COL_CANDIDATES if c in df.columns), None)
    value_col = next(
        (c for c in VALUE_COL_CANDIDATES if c in df.columns and df[c].notna().any()),
        None
    )
    if value_col is None:
        numeric_cols = [c for c in df.select_dtypes(include='number').columns if c != time_col]
        numeric_cols = [c for c in numeric_cols if df[c].notna().any()]
        if numeric_cols:
            value_col = max(numeric_cols, key=lambda c: df[c].notna().sum())
            if verbose:
                print(f'      [!] Guessing hydrograph value column: \'{value_col}\' '
                      f'(not in known candidates {VALUE_COL_CANDIDATES}) -- verify this is '
                      f'really the sensor reading. Columns available: {list(df.columns)}')
    return time_col, value_col

def build_hydrograph_data_uri(df, sensor_label='Sensor', figsize=(4.2, 2.0), chart_type='line'):
    """
    Renders a small hydrograph (value over time, peak marked in red) from
    a downloaded sensor DataFrame and returns it as a base64 PNG data URI
    ready to embed directly in a Folium popup's HTML (no external file,
    no CDN dependency -- works completely offline once the map is saved).
    Returns None if the DataFrame is empty or no usable time/value column
    is found.
    """
    if df is None or df.empty:
        return None
    time_col, value_col = _detect_time_value_cols(df)
    if time_col is None or value_col is None:
        return None

    plot_df = df[[time_col, value_col]].copy()
    # USGS/NWS-LID data's 'datetime' column is a raw ISO-8601 STRING from
    # the API JSON, never parsed to an actual datetime dtype (unlike IFC's
    # 'validTime', which is parsed at fetch time). Left as strings,
    # matplotlib treats each unique timestamp as its own text category
    # instead of a continuous time axis -- with ~15-minute USGS data over
    # a multi-day episode, that's hundreds of overlapping tick labels
    # rendering as an unreadable black smear. This parses it here, at the
    # charting layer only, so the raw CSV export (which some downstream
    # use might prefer as the original API string) is untouched.
    plot_df[time_col] = pd.to_datetime(plot_df[time_col], format='mixed', errors='coerce', utc=True).dt.tz_localize(None)
    plot_df[value_col] = pd.to_numeric(plot_df[value_col], errors='coerce')
    plot_df = plot_df.dropna().sort_values(time_col)
    if plot_df.empty:
        return None

    fig, ax = plt.subplots(figsize=figsize, dpi=100)
    if chart_type == 'bar':
        # Appropriate for precipitation/accumulation-style data (e.g.
        # hydrostation rain), where a hyetograph convention (bars) is the
        # standard way to show it -- a connected line reads as a
        # continuous quantity, which accumulated rain isn't.
        ax.bar(plot_df[time_col], plot_df[value_col], width=0.01, color='#3182bd')
    else:
        ax.plot(plot_df[time_col], plot_df[value_col], color='#08519c', linewidth=1.3)
    peak_idx  = plot_df[value_col].idxmax()
    peak_time = plot_df.loc[peak_idx, time_col]
    peak_val  = plot_df.loc[peak_idx, value_col]
    ax.scatter([peak_time], [peak_val], color='#e31a1c', zorder=5, s=25)
    ax.annotate(f'Peak: {peak_val:.2f}', xy=(peak_time, peak_val),
                xytext=(4, 4), textcoords='offset points', fontsize=7, color='#e31a1c')
    ax.set_title(sensor_label, fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=6)
    plt.setp(ax.get_yticklabels(), fontsize=6)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    return f'data:image/png;base64,{encoded}'

def build_hydrostation_chart_data_uri(df, sensor_label='Hydrostation', figsize=(4.4, 2.6)):
    """
    Hydrostations report multiple parameter types (rain, wind, soil,
    well, groundwell, stage) tagged via a 'parameter_type' column, unlike
    river/USGS sensors which report one value stream -- so a plain
    hydrograph doesn't fit them well. Confirmed against live data: many
    hydrostations report rain/soil/well/wind but NOT stage at all (stage
    is really the river sensors' domain, not hydrostations' -- this isn't
    missing/broken data, it's a genuinely different sensor category), so
    the single-panel rain fallback below is the common case, not an edge
    case. This builds the standard hydrology combo plot (rainfall bars on
    top, stage hydrograph with peak marked below, shared time axis) only
    when a station happens to report both. Falls back to a single-panel
    chart of whatever parameter type IS present otherwise, and to
    build_hydrograph_data_uri's generic behavior if there's no
    'parameter_type' column at all. Returns a base64 PNG data URI, or
    None if nothing plottable is found.
    """
    if df is None or df.empty or 'parameter_type' not in df.columns:
        return build_hydrograph_data_uri(df, sensor_label=sensor_label, figsize=(4.2, 2.0))

    stage_df = df[df['parameter_type'] == 'stage']
    rain_df  = df[df['parameter_type'] == 'rain']
    has_stage = not stage_df.empty
    has_rain  = not rain_df.empty

    if not has_stage and not has_rain:
        # Neither hydrologically primary parameter is present -- fall
        # back to whichever parameter type has the most records.
        if df['parameter_type'].notna().any():
            top_type = df['parameter_type'].value_counts().idxmax()
            return build_hydrograph_data_uri(
                df[df['parameter_type'] == top_type],
                sensor_label=f'{sensor_label} ({top_type})', figsize=(4.2, 2.0)
            )
        return None

    if has_stage and not has_rain:
        return build_hydrograph_data_uri(stage_df, sensor_label=f'{sensor_label} (stage)', figsize=(4.2, 2.0))
    if has_rain and not has_stage:
        return build_hydrograph_data_uri(rain_df, sensor_label=f'{sensor_label} (rain)', figsize=(4.2, 2.0), chart_type='bar')

    # Both present -- build the combo hyetograph (top) + hydrograph (bottom)
    stage_time, stage_val = _detect_time_value_cols(stage_df, verbose=False)
    rain_time, rain_val   = _detect_time_value_cols(rain_df, verbose=False)
    if stage_time is None or stage_val is None or rain_time is None or rain_val is None:
        return None

    s = stage_df[[stage_time, stage_val]].copy()
    s[stage_val] = pd.to_numeric(s[stage_val], errors='coerce')
    s = s.dropna().sort_values(stage_time)
    r = rain_df[[rain_time, rain_val]].copy()
    r[rain_val] = pd.to_numeric(r[rain_val], errors='coerce')
    r = r.dropna().sort_values(rain_time)
    if s.empty or r.empty:
        return None

    fig, (ax_rain, ax_stage) = plt.subplots(
        2, 1, figsize=figsize, dpi=100, sharex=True,
        gridspec_kw={'height_ratios': [1, 2]}
    )
    ax_rain.bar(r[rain_time], r[rain_val], width=0.02, color='#3182bd')
    ax_rain.invert_yaxis()  # hyetograph convention -- rain bars hang down from the top
    ax_rain.set_ylabel('Rain', fontsize=6)
    ax_rain.tick_params(labelsize=6)
    ax_rain.grid(alpha=0.3)

    ax_stage.plot(s[stage_time], s[stage_val], color='#08519c', linewidth=1.3)
    peak_idx  = s[stage_val].idxmax()
    peak_time = s.loc[peak_idx, stage_time]
    peak_val  = s.loc[peak_idx, stage_val]
    ax_stage.scatter([peak_time], [peak_val], color='#e31a1c', zorder=5, s=25)
    ax_stage.annotate(f'Peak: {peak_val:.2f}', xy=(peak_time, peak_val),
                       xytext=(4, 4), textcoords='offset points', fontsize=7, color='#e31a1c')
    ax_stage.set_ylabel('Stage', fontsize=6)
    ax_stage.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.setp(ax_stage.get_xticklabels(), rotation=40, ha='right', fontsize=6)
    ax_stage.tick_params(labelsize=6)
    ax_stage.grid(alpha=0.3)

    fig.suptitle(sensor_label, fontsize=8)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    return f'data:image/png;base64,{encoded}'

def build_usgs_chart_data_uri(df, sensor_label='USGS Gauge', figsize=(4.2, 2.2)):
    """
    USGS gauges can report both discharge (parameter code 00060) and
    gage height / stage (00065) for the same site -- fetch_usgs_sensor_data
    makes a best-effort attempt to fetch both. When both are present here
    (detected via USGS's own standard 'parameter' text, e.g. 'Gage height,
    feet' vs 'Discharge, cubic feet per second' -- these are stable,
    well-documented USGS naming conventions, not a guess like some of the
    IFC field names elsewhere in this pipeline), this draws the primary
    series (usually discharge) as a solid line with its peak marked, and
    overlays stage height as a dashed line on a secondary y-axis (since
    cfs and ft are wildly different scales -- a shared axis would flatten
    one of them to a flat line). Falls back to the plain single-line
    hydrograph if only one parameter is present, or if 'parameter' isn't
    a column at all. Returns a base64 PNG data URI, or None if nothing
    plottable is found.
    """
    if df is None or df.empty or 'parameter' not in df.columns:
        return build_hydrograph_data_uri(df, sensor_label=sensor_label)

    is_stage = df['parameter'].astype(str).str.contains('gage height', case=False, na=False)
    stage_df   = df[is_stage]
    primary_df = df[~is_stage]

    if stage_df.empty or primary_df.empty:
        # Only one parameter available -- plain single-line hydrograph,
        # exactly the prior behavior for USGS gauges.
        return build_hydrograph_data_uri(df, sensor_label=sensor_label)

    p_time, p_val = _detect_time_value_cols(primary_df, verbose=False)
    s_time, s_val = _detect_time_value_cols(stage_df, verbose=False)
    if p_time is None or p_val is None or s_time is None or s_val is None:
        return build_hydrograph_data_uri(primary_df, sensor_label=sensor_label)

    p = primary_df[[p_time, p_val]].copy()
    p[p_time] = pd.to_datetime(p[p_time], format='mixed', errors='coerce', utc=True).dt.tz_localize(None)
    p[p_val]  = pd.to_numeric(p[p_val], errors='coerce')
    p = p.dropna().sort_values(p_time)

    s = stage_df[[s_time, s_val]].copy()
    s[s_time] = pd.to_datetime(s[s_time], format='mixed', errors='coerce', utc=True).dt.tz_localize(None)
    s[s_val]  = pd.to_numeric(s[s_val], errors='coerce')
    s = s.dropna().sort_values(s_time)

    if p.empty and s.empty:
        return None
    if p.empty:
        return build_hydrograph_data_uri(stage_df, sensor_label=sensor_label)
    if s.empty:
        return build_hydrograph_data_uri(primary_df, sensor_label=sensor_label)

    fig, ax = plt.subplots(figsize=figsize, dpi=100)
    ax.plot(p[p_time], p[p_val], color='#08519c', linewidth=1.3, label='Discharge')
    peak_idx  = p[p_val].idxmax()
    peak_time = p.loc[peak_idx, p_time]
    peak_val  = p.loc[peak_idx, p_val]
    ax.scatter([peak_time], [peak_val], color='#e31a1c', zorder=5, s=25)
    ax.annotate(f'Peak: {peak_val:.2f}', xy=(peak_time, peak_val),
                xytext=(4, 4), textcoords='offset points', fontsize=7, color='#e31a1c')
    ax.set_ylabel('Discharge', fontsize=6, color='#08519c')
    ax.tick_params(axis='y', labelsize=6, labelcolor='#08519c')

    ax2 = ax.twinx()
    ax2.plot(s[s_time], s[s_val], color='#555555', linewidth=1.1, linestyle='--', label='Stage height')
    ax2.set_ylabel('Stage (ft)', fontsize=6, color='#555555')
    ax2.tick_params(axis='y', labelsize=6, labelcolor='#555555')

    ax.set_title(sensor_label, fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=6)
    ax.grid(alpha=0.3)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=5, loc='upper left', framealpha=0.7)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    return f'data:image/png;base64,{encoded}'

print('✓ Sensor helpers defined.')

In [ ]:
# ── Folium map builder ─────────────────────────────────────────────────────────

def fetch_huc8_geojson(huc8_code):
    """
    Fetches the actual HUC-8 polygon from the USGS NHD REST API as GeoJSON.
    Returns a dict (GeoJSON feature) or None.
    """
    url = 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query'
    params = {
        'where':        f"huc8='{huc8_code}'",
        'outFields':    'huc8,name',
        'f':            'geojson',
        'outSR':        '4326',
        'returnGeometry': 'true',
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        features = r.json().get('features', [])
        return features[0] if features else None
    except Exception as e:
        print(f'  ⚠️  Could not fetch HUC-8 polygon for {huc8_code}: {e}')
        return None


def build_episode_map(episode_id, episode_rows,
                      hwms_df=None,
                      ifc_sensors_df=None,
                      ifc_stations_df=None,
                      usgs_sensors_df=None,
                      inundation_gdf=None,
                      precip_total_da=None,
                      sensor_hydrographs=None):
    min_lat = episode_rows['BEGIN_LAT'].min()
    max_lat = episode_rows['END_LAT'].max()
    min_lon = episode_rows['BEGIN_LON'].min()
    max_lon = episode_rows['END_LON'].max()
    center  = [(min_lat+max_lat)/2, (min_lon+max_lon)/2]

    m = folium.Map(location=center, zoom_start=8, tiles='CartoDB positron')

    # 1. Episode bounding box
    folium.Rectangle(
        bounds=[[min_lat, min_lon],[max_lat, max_lon]],
        color='#1f78b4', weight=2.5, fill=True,
        fill_color='#1f78b4', fill_opacity=0.06,
        popup=f'NOAA Episode: {episode_id}', tooltip='Episode bounding box'
    ).add_to(m)

    # 2. IFC inundation extents -- the actual KMZ polygons fetched in step 3
    # (this replaces a previous WmsTileLayer pointed at a WMS endpoint that,
    # on inspection, IFC does not appear to publicly expose -- IFC serves
    # these maps as per-stage KML/KMZ files, which is what step 3 downloads).
    if inundation_gdf is not None and not inundation_gdf.empty:
        n_extents  = inundation_gdf['extent_name'].nunique() if 'extent_name' in inundation_gdf.columns else len(inundation_gdf)
        inund_group = folium.FeatureGroup(name=f'IFC Inundation Maps ({n_extents} extents)', show=False)
        # IFC's raw KML-derived flood extent polygons carry far more vertex
        # density than is visible at any web-map zoom (tens of thousands of
        # points per polygon isn't unusual), which bloats the saved HTML
        # file to tens of MB and can make the browser sluggish. Simplifying
        # by ~11m (0.0001 deg) cuts vertex count by ~90-95% with a <0.02%
        # area change -- imperceptible, since these are already generalized
        # hydrologic-model outputs, not survey-grade boundaries. This only
        # affects the MAP DISPLAY copy; the exported GeoPackage still uses
        # the original full-precision inundation_gdf, untouched here.
        SIMPLIFY_TOLERANCE_DEG = 0.0001
        for _, row in inundation_gdf.iterrows():
            extent_name = row.get('extent_name', 'inundation')
            is_freq     = str(extent_name).startswith('flood_')
            colour      = '#08519c' if is_freq else '#3182bd'
            simplified_geom = row.geometry.simplify(SIMPLIFY_TOLERANCE_DEG, preserve_topology=True)
            folium.GeoJson(
                simplified_geom.__geo_interface__,
                style_function=lambda x, c=colour: {
                    'color': c, 'weight': 1, 'fillColor': c, 'fillOpacity': 0.25
                },
                tooltip=str(extent_name)
            ).add_to(inund_group)
        inund_group.add_to(m)
        print(f'  ✓ Plotted {len(inundation_gdf)} inundation polygon(s) across {n_extents} extent(s) (simplified for display)')
    else:
        print('  [*] No inundation polygons to plot (none fetched in step 3)')

    # 3. HUC-8 watersheds — fetched as real GeoJSON polygons from USGS REST API
    huc8_codes = [h for h in episode_rows['HUC8_clean'].dropna().unique() if h != '00000000']
    huc8_name_map = dict(zip(
        episode_rows.drop_duplicates('HUC8_clean')['HUC8_clean'],
        episode_rows.drop_duplicates('HUC8_clean')['NAME']
    ))
    palette = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
               '#42d4f4','#f032e6','#bfef45','#fabed4','#469990']

    if huc8_codes:
        print(f'  Fetching {len(huc8_codes)} HUC-8 polygon(s) from USGS REST API...')
        huc8_group = folium.FeatureGroup(name=f'HUC-8 Watersheds ({len(huc8_codes)})', show=True)
        for i, huc in enumerate(huc8_codes):
            colour  = palette[i % len(palette)]
            basin_name = huc8_name_map.get(huc, huc)
            feature = fetch_huc8_geojson(huc)
            if feature:
                folium.GeoJson(
                    feature,
                    name=f'HUC-8: {basin_name}',
                    style_function=lambda x, c=colour: {
                        'color': c, 'weight': 2.5,
                        'fillColor': c, 'fillOpacity': 0.12
                    },
                    tooltip=folium.GeoJsonTooltip(fields=['huc8','name'], aliases=['HUC-8','Basin'])
                ).add_to(huc8_group)
                print(f'    ✓ {basin_name} ({huc})')
            else:
                print(f'    ⚠️  No polygon returned for {huc}')
        huc8_group.add_to(m)

    # 4. USGS High-Water Marks
    if hwms_df is not None and not hwms_df.empty:
        lat_col = 'latitude_dd'  if 'latitude_dd'  in hwms_df.columns else 'latitude'
        lon_col = 'longitude_dd' if 'longitude_dd' in hwms_df.columns else 'longitude'
        hwm_group = folium.FeatureGroup(name=f'USGS High-Water Marks ({len(hwms_df)})', show=True)
        for _, row in hwms_df.iterrows():
            try:
                lat, lon = float(row[lat_col]), float(row[lon_col])
            except (ValueError, TypeError, KeyError):
                continue
            photo_urls = get_hwm_photo_urls(row)
            # Field names confirmed against a live STNServices/HWMs.json
            # response (2026): snake_case (hwm_id, elev_ft, waterbody...),
            # not the camelCase names (hwmID, elevFt...) this originally
            # used, which matched nothing and produced all-N/A popups.
            # Note hwm_type_id/hwm_quality_id are only numeric codes here --
            # the response has no nested name lookup for them, so they're
            # shown as codes rather than invented labels.
            survey_date = str(row.get('survey_date', ''))[:10] or 'N/A'
            hwm_html = (
                f"<b>USGS HWM</b><br>Waterbody: {row.get('waterbody', 'N/A')}<br>"
                f"Location: {row.get('hwm_locationdescription', 'N/A')}<br>"
                f"Elev: {row.get('elev_ft', 'N/A')} ft<br>Environment: {row.get('hwm_environment', 'N/A')}<br>"
                f"Type code: {row.get('hwm_type_id', 'N/A')} | Quality code: {row.get('hwm_quality_id', 'N/A')}<br>"
                f"Survey date: {survey_date}<br>ID: {row.get('hwm_id', 'N/A')}"
            )
            if photo_urls:
                # These are live links to USGS's own STN file server, not
                # embedded/offline like the sensor hydrographs -- viewing
                # them later requires an internet connection.
                for url in photo_urls:
                    hwm_html += f'<br><img src="{url}" width="260" loading="lazy">'
            folium.CircleMarker(
                location=[lat, lon], radius=5,
                color='#e31a1c', fill=True, fill_color='#fb9a99', fill_opacity=0.85,
                popup=folium.Popup(hwm_html, max_width=300 if photo_urls else 220),
                tooltip='USGS HWM'
            ).add_to(hwm_group)
        hwm_group.add_to(m)

    sensor_hydrographs = sensor_hydrographs or {}

    def _popup_html(header, code, extra_line, hydrograph=True):
        """Builds popup HTML, embedding the sensor's hydrograph image
        (if one was built for it) beneath its metadata."""
        html = f"<b>{header}</b><br>ID: {code}<br>{extra_line}"
        uri = sensor_hydrographs.get(code) if hydrograph else None
        if uri:
            html += f'<br><img src="{uri}" width="380">'
        return html

    # 5. IFC River Sensors
    if ifc_sensors_df is not None and not ifc_sensors_df.empty:
        ifc_s_group = folium.FeatureGroup(name=f'IFC River Sensors ({len(ifc_sensors_df)})', show=True)
        plotted = 0
        for _, row in ifc_sensors_df.iterrows():
            lat, lon = get_lat_lon(row, LAT_COLS, LON_COLS)
            if lat is None or lon is None: continue
            code = get_sensor_code(row, 'ifc_river')
            has_chart = code in sensor_hydrographs
            folium.CircleMarker(
                location=[lat, lon], radius=7,
                color='#2ca02c', fill=True, fill_color='#98df8a', fill_opacity=0.9,
                popup=folium.Popup(
                    _popup_html('IFC River Sensor', code, f"HUC-8: {row.get('HUC8','N/A')}"),
                    max_width=420 if has_chart else 220),
                tooltip='IFC River Sensor'
            ).add_to(ifc_s_group)
            plotted += 1
        ifc_s_group.add_to(m)
        print(f'  ✓ Plotted {plotted}/{len(ifc_sensors_df)} IFC river sensors')

    # 6. IFC Hydrostations
    if ifc_stations_df is not None and not ifc_stations_df.empty:
        ifc_h_group = folium.FeatureGroup(name=f'IFC Hydrostations ({len(ifc_stations_df)})', show=True)
        plotted = 0
        for _, row in ifc_stations_df.iterrows():
            lat, lon = get_lat_lon(row, LAT_COLS, LON_COLS)
            if lat is None or lon is None: continue
            code = get_sensor_code(row, 'ifc_hydrostation')
            has_chart = code in sensor_hydrographs
            folium.CircleMarker(
                location=[lat, lon], radius=7,
                color='#ff7f0e', fill=True, fill_color='#ffbb78', fill_opacity=0.9,
                popup=folium.Popup(
                    _popup_html('IFC Hydrostation', code, f"HUC-8: {row.get('HUC8','N/A')}"),
                    max_width=420 if has_chart else 220),
                tooltip='IFC Hydrostation'
            ).add_to(ifc_h_group)
            plotted += 1
        ifc_h_group.add_to(m)
        print(f'  ✓ Plotted {plotted}/{len(ifc_stations_df)} IFC hydrostations')

    # 7. USGS Gauges
    if usgs_sensors_df is not None and not usgs_sensors_df.empty:
        usgs_group = folium.FeatureGroup(name=f'USGS Gauges ({len(usgs_sensors_df)})', show=True)
        plotted = 0
        for _, row in usgs_sensors_df.iterrows():
            lat, lon = get_lat_lon(row, LAT_COLS, LON_COLS)
            if lat is None or lon is None: continue
            code = get_sensor_code(row, 'usgs')
            has_chart = code in sensor_hydrographs
            folium.CircleMarker(
                location=[lat, lon], radius=7,
                color='#9467bd', fill=True, fill_color='#c5b0d5', fill_opacity=0.9,
                popup=folium.Popup(
                    _popup_html('USGS Gauge', code, f"HUC-8: {row.get('HUC8','N/A')}"),
                    max_width=420 if has_chart else 220),
                tooltip='USGS Gauge'
            ).add_to(usgs_group)
            plotted += 1
        usgs_group.add_to(m)
        print(f'  ✓ Plotted {plotted}/{len(usgs_sensors_df)} USGS gauges')

    # 9. NOAA MRMS total storm precipitation (accumulated over the episode)
    if precip_total_da is not None:
        rgba, bounds, colormap = precip_grid_to_overlay(precip_total_da)
        precip_group = folium.FeatureGroup(name='MRMS Total Precipitation (mm)', show=True)
        folium.raster_layers.ImageOverlay(
            image=rgba, bounds=bounds, opacity=0.8,
            name='MRMS Total Precipitation (mm)'
        ).add_to(precip_group)
        precip_group.add_to(m)
        colormap.caption = 'MRMS total precipitation, episode window (mm)'
        colormap.add_to(m)
        print(f'  ✓ MRMS precipitation overlay added ({precip_total_da.shape[0]}x{precip_total_da.shape[1]} grid)')

    folium.LayerControl(collapsed=False).add_to(m)
    return m

print('✓ Folium map builder defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MASTER PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

df = load_noaa_events(NOAA_EVENTS_FILE)

# ── Episode selection ──────────────────────────────────────────────────────────
print('==================================================')
print(' NOAA Episode -> Master Pipeline')
print('==================================================')

episodes_summary = df.groupby('NEW_EPISODE_ID').agg(
    event_types =('EVENT_TYPE', lambda x: ', '.join(x.unique())),
    counties    =('CZ_NAME',   lambda x: ', '.join(x.unique())),
    event_count =('EVENT_ID',  'count')
).reset_index()

print('\nSample Episodes:\n')
print(episodes_summary[['NEW_EPISODE_ID','event_types','counties','event_count']].head(10).to_string(index=False))
print('\n--------------------------------------------------')

user_input = input(f"\nEnter NEW_EPISODE_ID (or Enter for '{DEFAULT_EPISODE_ID}'): ").strip()
if not user_input:
    user_input = DEFAULT_EPISODE_ID

episode_rows = df[df['NEW_EPISODE_ID'].astype(str) == str(user_input)]
if episode_rows.empty:
    print(f"Episode '{user_input}' not found.")
    raise SystemExit

# ── Metadata ───────────────────────────────────────────────────────────────────
fips_list    = episode_rows['FIPS_5'].unique().tolist()
county_names = episode_rows['CZ_NAME'].unique().tolist()
huc8_names   = episode_rows['NAME'].unique().tolist()
huc8_codes   = [h for h in episode_rows['HUC8_clean'].unique().tolist() if h != '00000000']
min_date     = episode_rows['BEGIN_DT'].min()
max_date     = episode_rows['END_DT'].max()
min_lat      = episode_rows['BEGIN_LAT'].min()
max_lat      = episode_rows['END_LAT'].max()
min_lon      = episode_rows['BEGIN_LON'].min()
max_lon      = episode_rows['END_LON'].max()
safe_id      = user_input.replace('/', '_')
start_time   = min_date - timedelta(hours=3)
end_time     = max_date + timedelta(hours=3)

print('\n--------------------------------------------------')
print('Selected Episode Summary:')
print(f' • Episode ID:    {user_input}')
print(f" • Event Types:   {', '.join(episode_rows['EVENT_TYPE'].unique())}")
print(f" • Counties ({len(fips_list)}): {', '.join(county_names)} (FIPS: {', '.join(fips_list)})")
print(f" • HUC-8 Basins:  {', '.join(huc8_names)}")
print(f' • Date Window:   {min_date.strftime("%Y-%m-%d %H:%M")} to {max_date.strftime("%Y-%m-%d %H:%M")}')
print(f' • Bounding Box:  Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]')
print('--------------------------------------------------\n')

# ── 1. FEMA Claims ─────────────────────────────────────────────────────────────
print('[1/5] Querying OpenFEMA NFIP Claims...')
claims_df = query_fema_claims_multi_fips(fips_list=fips_list, start_date=min_date, end_date=max_date)
if claims_df.empty:
    print('  No FEMA NFIP claims found.')
else:
    print(f'  Found {len(claims_df)} matching claim(s)!\n')
    cols_show = [c for c in ['dateOfLoss','countyCode','amountPaidOnBuildingClaim','amountPaidOnContentsClaim'] if c in claims_df.columns]
    print(claims_df[cols_show].head(10).to_string(index=False))
    if 'amountPaidOnBuildingClaim' in claims_df.columns:
        tot_b = claims_df['amountPaidOnBuildingClaim'].sum()
        tot_c = claims_df['amountPaidOnContentsClaim'].sum() if 'amountPaidOnContentsClaim' in claims_df.columns else 0
        print('\nFinancial Totals for Matching Episode Claims:')
        print(f' • Building Payouts: ${tot_b:,.2f}')
        print(f' • Contents Payouts: ${tot_c:,.2f}')
        print(f' • Total Payouts:    ${(tot_b+tot_c):,.2f}')

print('\n--------------------------------------------------')

# ── 2. USGS High-Water Marks ───────────────────────────────────────────────────
print('[2/5] Querying USGS STN High-Water Marks...')
print(f'  Bounding Box: Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]')
print(f'  Date Window:  {min_date.strftime("%Y-%m-%d")} to {max_date.strftime("%Y-%m-%d")}')
hwms_df = query_usgs_hwms(min_lat=min_lat, max_lat=max_lat, min_lon=min_lon, max_lon=max_lon,
                           start_date=min_date, end_date=max_date)
if hwms_df.empty:
    print('  No matching USGS High-Water Marks found.')
else:
    print(f'\n  Found {len(hwms_df)} matching USGS High-Water Mark(s)!\n')
    cols_hwm = [c for c in ['hwmID','eventName','latitude_dd','longitude_dd','elevFt','hwmQualityName','hwmTypeName'] if c in hwms_df.columns]
    print(hwms_df[cols_hwm].head(10).to_string(index=False))

print('\n--------------------------------------------------')

# ── 3. IFC Inundation Maps ─────────────────────────────────────────────────────
print('[3/5] Fetching IFC Inundation Maps...')
community_lookup = load_ifis_community_lookup()
target_ifis_ids, matched_names = match_communities_for_episode(
    episode_rows, community_lookup, county_to_cities=COUNTY_TO_CITIES
)
if matched_names:
    print(f"  [>] Matched communities: {', '.join(sorted(matched_names))} "
          f"-> IFIS IDs {sorted(target_ifis_ids)}")
if not target_ifis_ids:
    print('  [*] No community matched — defaulting to Iowa City')
    target_ifis_ids.add(community_lookup.get('IOWA CITY', 598))

all_kmz_items = set()
for ifis_id in target_ifis_ids:
    all_kmz_items.update(get_ifis_kmz_items(ifis_id))

gdfs                 = []
kmz_zip_buf          = None
gpkg_file            = None
inundation_gdf_4326  = None

if all_kmz_items:
    gpkg_file = f'episode_{safe_id}_inundation_layers.gpkg'
    print(f'  Fetching {len(all_kmz_items)} KMZ files...')
    kmz_zip_buf = io.BytesIO()
    with zipfile.ZipFile(kmz_zip_buf, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:
        for kmz_url, extent_name in sorted(all_kmz_items, key=lambda x: x[1]):
            raw_fn  = kmz_url.split('/')[-1]
            desc_fn = f'{extent_name}_{raw_fn}'
            try:
                r = requests.get(kmz_url, timeout=15)
                if r.status_code == 200:
                    zf.writestr(desc_fn, r.content)
                    print(f'    ✓ {desc_fn}')
                    gdf = convert_kmz_bytes_to_geodataframe(r.content, extent_name)
                    if gdf is not None and not gdf.empty:
                        gdfs.append(gdf)
            except Exception as e:
                print(f'    [-] Error downloading {kmz_url}: {e}')
    if gdfs:
        combined_gdf = pd.concat(gdfs, ignore_index=True)
        drop_cols    = [c for c in ['Description','StyleMap','styleUrl'] if c in combined_gdf.columns]
        combined_gdf = combined_gdf.drop(columns=drop_cols)
        combined_gdf.to_file(gpkg_file, driver='GPKG')
        print(f'  [✓] GeoPackage saved: {gpkg_file} (EPSG:3418)')
        # Folium/Leaflet requires WGS84 (EPSG:4326) -- keep a separate
        # reprojected copy just for the map; the saved GPKG stays EPSG:3418.
        inundation_gdf_4326 = combined_gdf.to_crs(epsg=4326)
else:
    print('  [-] No inundation KMZ files found.')
    inundation_gdf_4326 = None

print('\n--------------------------------------------------')

# ── 4. Sensor matching + time-series download ─────────────────────────
print('[4/5] Matching sensors to episode HUC-8 watersheds...')
ifc_sensors       = pd.read_csv(IFC_SENSORS_FILE)
ifc_hydrostations = pd.read_csv(IFC_HYDROSTATIONS_FILE)
usgs_sensors      = pd.read_csv(USGS_SENSORS_FILE)

matched_ifc_sensors  = ifc_sensors[normalize_huc(ifc_sensors['HUC8']).isin(huc8_codes)]
matched_ifc_stations = ifc_hydrostations[normalize_huc(ifc_hydrostations['HUC8']).isin(huc8_codes)]
matched_usgs_sensors = usgs_sensors[normalize_huc(usgs_sensors['HUC8']).isin(huc8_codes)]

print(f'  • IFC River Sensors:  {len(matched_ifc_sensors)}')
print(f'  • IFC Hydrostations:  {len(matched_ifc_stations)}')
print(f'  • USGS Sensors:       {len(matched_usgs_sensors)}')

# Sensor time series are fetched here -- BEFORE the map is built -- rather
# than only at ZIP-export time, so each sensor's hydrograph can be
# embedded in its map popup. This means time series now download on
# every run, not just when you opt into the final ZIP export (a real
# behavior change from before). The ZIP-export step later reuses this
# same dict instead of re-fetching, so nothing gets downloaded twice.
total_sensors = len(matched_ifc_sensors) + len(matched_ifc_stations) + len(matched_usgs_sensors)
sensor_timeseries = {}   # {(folder, code): DataFrame}
sensor_hydrographs = {}  # {code: data URI} -- built alongside, for map popups

if total_sensors > 0:
    print(f'\n  Downloading sensor time series ({total_sensors} sensors, {MAX_WORKERS} threads)...')
    sensor_tasks = []
    for _, row in matched_ifc_sensors.iterrows():
        code = get_sensor_code(row, 'ifc_river')
        sensor_tasks.append(('sensors/ifc_river', code, fetch_ifc_sensor_data, (row, start_time, end_time)))
    for _, row in matched_ifc_stations.iterrows():
        code = get_sensor_code(row, 'ifc_hydrostation')
        sensor_tasks.append(('sensors/ifc_hydrostations', code, fetch_ifc_hydrostation_data, (row, start_time, end_time)))
    for _, row in matched_usgs_sensors.iterrows():
        code = get_sensor_code(row, 'usgs')
        sensor_tasks.append(('sensors/usgs', code, fetch_usgs_sensor_data, (row, start_time, end_time)))

    sensor_success_counts = {'sensors/ifc_river': 0, 'sensors/ifc_hydrostations': 0, 'sensors/usgs': 0}
    sensor_total_counts   = {'sensors/ifc_river': 0, 'sensors/ifc_hydrostations': 0, 'sensors/usgs': 0}
    for folder, *_ in sensor_tasks:
        sensor_total_counts[folder] += 1

    completed = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_task = {
            executor.submit(fetch_with_retry, fn, args): (folder, code)
            for folder, code, fn, args in sensor_tasks
        }
        for future in as_completed(future_to_task):
            folder, code = future_to_task[future]
            completed += 1
            try:
                result_df = future.result()
            except Exception as exc:
                print(f'      ⚠️ Error {folder}/{code}: {exc}')
                result_df = pd.DataFrame()
            if result_df is not None and not result_df.empty:
                sensor_success_counts[folder] += 1
                sensor_timeseries[(folder, code)] = result_df
                # Hydrostations get the rain+stage combo chart (they report
                # multiple parameter types); river/USGS sensors get the
                # plain single-value hydrograph.
                if folder == 'sensors/ifc_hydrostations':
                    chart_uri = build_hydrostation_chart_data_uri(result_df, sensor_label=code)
                elif folder == 'sensors/usgs':
                    chart_uri = build_usgs_chart_data_uri(result_df, sensor_label=code)
                else:
                    chart_uri = build_hydrograph_data_uri(result_df, sensor_label=code)
                if chart_uri:
                    sensor_hydrographs[code] = chart_uri
            if completed % 10 == 0 or completed == len(sensor_tasks):
                print(f'     ...{completed}/{len(sensor_tasks)} sensors processed')

    print('\n  📊 Sensor Collection Summary:')
    print(f"     • IFC River Sensors:  {sensor_success_counts['sensors/ifc_river']}/{sensor_total_counts['sensors/ifc_river']}")
    print(f"     • IFC Hydrostations:  {sensor_success_counts['sensors/ifc_hydrostations']}/{sensor_total_counts['sensors/ifc_hydrostations']}")
    print(f"     • USGS Sensors:       {sensor_success_counts['sensors/usgs']}/{sensor_total_counts['sensors/usgs']}")
    print(f'     • Hydrographs built:  {len(sensor_hydrographs)}/{len(sensor_timeseries)} (sensors with data but no usable value column are skipped -- see any [!] warnings above)')

print('\n--------------------------------------------------')

# ── 6. NOAA MRMS QPE precipitation ───────────────────────────────────────────
print('[5/5] Querying NOAA MRMS QPE precipitation...')

# NOAA Storm Events BEGIN_DATE_TIME/END_DATE_TIME are LOCAL time, not UTC,
# while MRMS grid files are timestamped in UTC. This converts using the
# CZ_TIMEZONE column if the NOAA events file has one (e.g. 'CST-6' -> UTC-6).
# If that column isn't present, times are treated as already UTC and a
# warning is printed -- for Iowa (Central Time) that fallback would be off
# by 5-6 hours, so verify the conversion against a known event's radar
# loop before trusting the resulting hourly series.
tz_offset_hours = None
if 'CZ_TIMEZONE' in episode_rows.columns:
    tz_vals = episode_rows['CZ_TIMEZONE'].dropna().unique()
    if len(tz_vals) > 0 and '-' in str(tz_vals[0]):
        try:
            tz_offset_hours = -int(str(tz_vals[0]).split('-')[-1])
        except ValueError:
            tz_offset_hours = None

if tz_offset_hours is not None:
    print(f'  [*] Converting local -> UTC using CZ_TIMEZONE={tz_vals[0]} (offset {tz_offset_hours}h)')
    mrms_start_utc = start_time - timedelta(hours=tz_offset_hours)
    mrms_end_utc   = end_time   - timedelta(hours=tz_offset_hours)
else:
    print('  [!] No usable CZ_TIMEZONE column found -- treating episode times as UTC.')
    print('      This is likely WRONG for Iowa (Central Time). Verify before trusting this series.')
    mrms_start_utc = start_time
    mrms_end_utc   = end_time

mrms_df = query_mrms_precipitation(
    start_date_utc=mrms_start_utc, end_date_utc=mrms_end_utc,
    min_lat=min_lat, max_lat=max_lat, min_lon=min_lon, max_lon=max_lon,
)
if not mrms_df.empty and mrms_df['mean_mm'].notna().any():
    total_mean_mm = mrms_df['mean_mm'].sum(skipna=True)
    n_hours = mrms_df['mean_mm'].notna().sum()
    print(f'  [✓] Episode total (mean-of-grid) precipitation: {total_mean_mm:.1f} mm across {n_hours} hour(s)')
else:
    print('  [-] No MRMS precipitation data retrieved for this episode')

# Spatial accumulation raster (for the map overlay and GeoTIFF/GeoPackage
# export) -- a second pass over the same hours, but keeping full grids
# instead of just bbox summary stats, then summed into one storm total.
#
# This is masked to the ACTUAL matched HUC-8 watershed shapes, not the
# flat episode bounding box: rain that falls outside a watershed's
# boundary doesn't contribute to that watershed's streamflow response,
# so a bounding box always includes irrelevant area (and can just as
# easily clip off part of an irregular watershed that pokes outside the
# box). Reuses fetch_huc8_geojson(), the same HUC-8 polygon fetch the
# map already does, so this doesn't introduce a new data source.
print('\n  Fetching watershed boundaries for precipitation masking...')
huc8_geoms = []
for huc in huc8_codes:
    feature = fetch_huc8_geojson(huc)
    if feature:
        huc8_geoms.append(shapely_shape(feature['geometry']))
watershed_union = unary_union(huc8_geoms) if huc8_geoms else None

if watershed_union is not None:
    w_min_lon, w_min_lat, w_max_lon, w_max_lat = watershed_union.bounds
    print(f'  [*] Watershed union bbox: Lat [{w_min_lat:.3f}, {w_max_lat:.3f}], '
          f'Lon [{w_min_lon:.3f}, {w_max_lon:.3f}] ({len(huc8_geoms)} basin(s))')
else:
    print('  [!] No watershed geometry available -- falling back to episode bounding box')
    w_min_lat, w_max_lat, w_min_lon, w_max_lon = min_lat, max_lat, min_lon, max_lon

print('  Building total-precipitation raster...')
precip_total_da, precip_count_da = compute_episode_precip_grid(
    start_date_utc=mrms_start_utc, end_date_utc=mrms_end_utc,
    min_lat=w_min_lat, max_lat=w_max_lat, min_lon=w_min_lon, max_lon=w_max_lon,
)

if precip_total_da is not None and watershed_union is not None:
    try:
        precip_total_da = _grid_to_180_lon(precip_total_da)
        precip_total_da = precip_total_da.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
        precip_total_da = precip_total_da.rio.write_crs('EPSG:4326')
        precip_total_da = precip_total_da.rio.clip(
            [shapely_mapping(watershed_union)], crs='EPSG:4326', drop=False
        )
        print(f'  [✓] Precipitation raster clipped to watershed boundaries')
    except Exception as e:
        print(f'  [!] Could not clip to watershed shape, using full bbox instead: {e}')

mrms_tif_file = None
if precip_total_da is not None:
    mrms_tif_file = f'episode_{safe_id}_mrms_precip_total.tif'
    save_precip_grid_geotiff(precip_total_da, mrms_tif_file)
    print(f'  [✓] GeoTIFF saved: {mrms_tif_file}')

    # Append the raster into the SAME GeoPackage as the IFC inundation
    # polygons where possible, so both live in one spatial file. If no
    # inundation polygons existed for this episode (gpkg_file is None),
    # give the raster its own GeoPackage instead of skipping it.
    mrms_gpkg_target = gpkg_file if gpkg_file else f'episode_{safe_id}_mrms_precip.gpkg'
    append_precip_grid_to_gpkg(precip_total_da, mrms_gpkg_target)
    if not gpkg_file:
        gpkg_file = mrms_gpkg_target  # so it gets picked up by the ZIP step below
else:
    print('  [-] No precipitation raster could be built for this episode')

print('\n--------------------------------------------------')

# ── Map ────────────────────────────────────────────────────────────────────────
print('\nBuilding interactive Folium map...')
m = build_episode_map(
    episode_id       = user_input,
    episode_rows     = episode_rows,
    hwms_df          = hwms_df          if not hwms_df.empty          else None,
    ifc_sensors_df   = matched_ifc_sensors  if not matched_ifc_sensors.empty  else None,
    ifc_stations_df  = matched_ifc_stations if not matched_ifc_stations.empty else None,
    usgs_sensors_df  = matched_usgs_sensors if not matched_usgs_sensors.empty else None,
    inundation_gdf   = inundation_gdf_4326,
    precip_total_da  = precip_total_da,
    sensor_hydrographs = sensor_hydrographs,
)
map_file = f'episode_{safe_id}_map.html'
m.save(map_file)
print(f"  ✓ Map saved to '{map_file}'")

print('\n--------------------------------------------------')

# ── Single download prompt ─────────────────────────────────────────────────────
# total_sensors is already computed back in step 4.

print(f"""
Ready to export. Here is what will be included in the ZIP:
  tabular/   — FEMA claims CSV {'✓' if not claims_df.empty else '(no data)'}
              — USGS HWMs CSV  {'✓' if not hwms_df.empty  else '(no data)'}
              — MRMS QPE precip CSV {'✓' if not mrms_df.empty and mrms_df['mean_mm'].notna().any() else '(no data)'}
  spatial/   — IFC inundation GeoPackage {'✓' if gpkg_file else '(no data)'}
             — IFC raw KMZ archive {'✓' if kmz_zip_buf else '(no data)'}
             — MRMS precip GeoTIFF {'✓' if mrms_tif_file else '(no data)'} (also appended as a raster table inside the GeoPackage, if that succeeded)
  sensors/   — IFC River Sensors ({len(matched_ifc_sensors)} sensors)
             — IFC Hydrostations ({len(matched_ifc_stations)} stations)
             — USGS Gauges ({len(matched_usgs_sensors)} gauges)
  maps/      — Interactive HTML map ✓
""")

do_download = input('Would you like to download all outputs as a single organized ZIP? (y/n): ').strip().lower()

if do_download not in ['y','yes']:
    print('Export skipped.')
else:
    master_zip = f'episode_{safe_id}_all_outputs.zip'
    print(f"\nBuilding '{master_zip}'...")

    with zipfile.ZipFile(master_zip, 'w', zipfile.ZIP_DEFLATED) as zf:

        # README
        readme = '\n'.join([
            f'Episode ID:    {user_input}',
            f"Event Types:   {', '.join(episode_rows['EVENT_TYPE'].unique())}",
            f"Counties:      {', '.join(county_names)}",
            f"HUC-8 Basins:  {', '.join(huc8_names)}",
            f'Date Window:   {min_date.strftime("%Y-%m-%d %H:%M")} to {max_date.strftime("%Y-%m-%d %H:%M")}',
            f'Bounding Box:  Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]',
            '',
            'Folder structure:',
            '  tabular/  — FEMA claims CSV, USGS HWM CSV, MRMS QPE precip CSV',
            '  spatial/  — IFC inundation GeoPackage (EPSG:3418, may also contain an MRMS raster table), raw KMZ archive, MRMS precip GeoTIFF',
            '  sensors/  — time-series CSVs per sensor (subfolders by type)',
            '  maps/     — interactive HTML map (open in any browser)',
            '',
            f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        ])
        zf.writestr('README.txt', readme)

        # tabular/
        if not claims_df.empty:
            zf.writestr(f'tabular/fema_claims_episode_{safe_id}.csv', claims_df.to_csv(index=False))
            print('  ✓ tabular/fema_claims')
        if not hwms_df.empty:
            zf.writestr(f'tabular/usgs_hwms_episode_{safe_id}.csv', hwms_df.to_csv(index=False))
            print('  ✓ tabular/usgs_hwms')
        if not mrms_df.empty and mrms_df['mean_mm'].notna().any():
            zf.writestr(f'tabular/mrms_qpe_precip_episode_{safe_id}.csv', mrms_df.to_csv(index=False))
            print('  ✓ tabular/mrms_qpe_precip')

        # spatial/
        if kmz_zip_buf:
            zf.writestr(f'spatial/episode_{safe_id}_inundation_maps.zip', kmz_zip_buf.getvalue())
            print('  ✓ spatial/inundation_maps.zip')
        if gpkg_file and os.path.exists(gpkg_file):
            zf.write(gpkg_file, f'spatial/{gpkg_file}')
            print(f'  ✓ spatial/{gpkg_file}')
        if mrms_tif_file and os.path.exists(mrms_tif_file):
            zf.write(mrms_tif_file, f'spatial/{mrms_tif_file}')
            print(f'  ✓ spatial/{mrms_tif_file}')

        # maps/
        if os.path.exists(map_file):
            zf.write(map_file, f'maps/{map_file}')
            print(f'  ✓ maps/{map_file}')

        # sensors/ -- reuses sensor_timeseries, already downloaded back in step 4
        # (before the map was built, so hydrograph popups had data to draw
        # from). Nothing gets re-fetched here anymore.
        if sensor_timeseries:
            print(f'\n  Writing {len(sensor_timeseries)} already-downloaded sensor time series to ZIP...')
            for (folder, code), result_df in sensor_timeseries.items():
                zf.writestr(f'{folder}/{code}.csv', result_df.to_csv(index=False))
            print(f'  ✓ {len(sensor_timeseries)} sensor CSVs written')

    print(f"\n✓ ZIP saved: '{master_zip}'")
    print('  Structure:')
    with zipfile.ZipFile(master_zip,'r') as zf:
        for name in sorted(zf.namelist()):
            print(f'    {name}')

In [ ]:
# Render map inline in Jupyter
m